# Unit 02: Matrices as Linear Maps

Use this notebook to connect the by-hand linear map exercises to PyTorch matrix multiplication. The key sentence for this unit is: a matrix represents a linear map once you know where the basis vectors go.

## Setup

Import PyTorch before starting. You will also need `torch.nn` when verifying against the real `nn.Linear`.

API you may need:

```python
import torch
from torch import nn
```

In [23]:
import torch
from torch import nn

# PyTorch Convention

We write the vectors as rows in our tensor.

Vector @ Matrix -> means weighted sum of rows of matrix (our vectors).

## By-Hand Problems

Write these out on paper or in a text cell before coding.

### Problem 1: Build a matrix from basis images

`T(e1) = (2, 1)`, `T(e2) = (0, 3)`. Write the matrix.

Math reminder:

```text
If e1 maps to v1 and e2 maps to v2, then the matrix has v1 and v2 as columns:

A = [T(e1)  T(e2)]
```

In [2]:
A = torch.tensor([[2, 1], [0, 3]])
print(A)

tensor([[2, 1],
        [0, 3]])


### Problem 2: Apply the matrix

Apply your matrix from Problem 1 to `(4, 5)`.

Math reminder:

```text
A @ x means: combine the columns of A using the coordinates of x.

If x = (4, 5), then A @ x = 4 * first_column + 5 * second_column.
```

In [3]:
v = torch.tensor([4, 5])
result = v @ A
print(result)

tensor([ 8, 19])


### Problem 3: 90-degree counterclockwise rotation

Write the matrix for a 90-degree counterclockwise rotation.

Derive it by asking where the standard basis vectors go:

```text
Where does e1 = (1, 0) go?
Where does e2 = (0, 1) go?
Put those two output vectors into the columns of the matrix.
```

In [5]:
# rotate basis 
A = torch.tensor([[0, 1], [-1, 0]])

### Problem 4: Reflection across y = x

Write the matrix for reflection across `y = x`.

Derive it by asking:

```text
Where does e1 = (1, 0) go after reflection across y = x?
Where does e2 = (0, 1) go after reflection across y = x?
Put those images into the columns of the matrix.
```

In [6]:
# swap the x and y
A = torch.tensor([[0, 1], [1, 0]])

## nn.Linear without nn.Linear

Goal: implement `nn.Linear` from scratch, then verify against the real one. Add shape comments throughout your code as you work.

##### What is nn.Linear?
A PyTorch module that wraps the affine transformation `y = x @ A.T + b` as a reusable, learnable layer.
It's the standard "fully connected" / "dense" layer in neural nets.

After linear = nn.Linear(3, 2), the layer has:

`linear.weight` : tensor of shape (out_features, in_features) = (2, 3)

`linear.bias` : tensor of shape (out_features,) = (2,)

Both randomly initialized by default.

### Setup dimensions and parameters

Pick small dimensions:

- `in_features = 3`
- `out_features = 2`
- `W`: shape `(in_features, out_features)`, random values
- `b`: shape `(out_features,)`, random values

API you may need:

```python
torch.randn(rows, cols)
torch.randn(n)
x.shape
```

In [10]:
in_features = 3
out_features = 2

W = torch.randn(in_features, out_features)
b = torch.randn(out_features,) # bias term
print(W)
print(b)

tensor([[ 0.2302,  1.2362],
        [ 0.7597, -1.2230],
        [-0.7996,  0.7273]])
tensor([ 0.9590, -0.3395])


### Step 1: Single-vector implementation

`x`: shape `(in_features,)`.

Implement `y = x @ W + b`.

Shape check to verify:

```text
x is (3,)
W is (3, 2)
x @ W is (2,)
b is (2,)
y is (2,)
```

API you may need:

```python
x @ W
x @ W + b
tensor.shape
```

In [11]:
x = torch.randn(in_features,)
print(x.shape)
print(W.shape)

torch.Size([3])
torch.Size([3, 2])


In [13]:
print(x @ W)

tensor([-1.2213,  0.4047])


In [12]:
y = x @ W + b
print(y)

tensor([-0.2623,  0.0652])


### Step 2: Batched implementation

`X`: shape `(batch_size, in_features)`, for example `batch_size = 5`.

Implement `Y = X @ W + b`.

Shape check to verify:

```text
X is (5, 3)
W is (3, 2)
X @ W is (5, 2)
b is (2,), broadcast across rows
Y is (5, 2)
```

API you may need:

```python
torch.randn(batch_size, in_features)
X @ W + b
Y.shape
```

In [17]:
batch_size = 5
# Create 5 rows of 3 features each
X = torch.randn(batch_size, in_features)
print(X)

tensor([[ 0.6555, -0.3622, -0.3315],
        [ 1.2669,  0.3653,  0.1811],
        [-0.6630,  0.2772,  0.7811],
        [-0.3366, -0.6555,  0.1710],
        [-1.3655,  0.3205, -0.4794]])


In [18]:
print(W)
# shape is (3, 2)
# each row in X gets compressed from 3 features to 2

tensor([[ 0.2302,  1.2362],
        [ 0.7597, -1.2230],
        [-0.7996,  0.7273]])


In [21]:
y = X @ W + b
print(y)
# we now have 5 2d vectors
# W acts as a linear map converting our 5 3d vectors to 5 2d vectors

tensor([[ 1.0998,  0.6727],
        [ 1.3834,  0.9116],
        [ 0.3924, -0.9301],
        [ 0.2468,  0.1703],
        [ 1.2715, -2.7682]])


### Step 3: Verify against nn.Linear


Create `linear = nn.Linear(in_features=3, out_features=2)`.

Heads-up: `linear.weight` has shape `(out_features, in_features) = (2, 3)`, which is the transpose of your `W`. PyTorch stores weights this way and internally computes `x @ A.T`.

Copy weights:

```text
linear.weight.data = W.T.clone()
linear.bias.data = b.clone()
```

Run `X` through both your code and `linear(X)`. Compare with `torch.allclose`.

API you may need:

```python
nn.Linear(in_features=3, out_features=2)
W.T
W.T.clone()
b.clone()
linear(X)
torch.allclose(Y_manual, Y_linear)
```

In [26]:
linear = nn.Linear(in_features=in_features, out_features=out_features)
linear.weight.data = W.T.clone()
linear.bias.data = b.clone()
print(linear)
print(linear.weight)
print(linear.bias)

Linear(in_features=3, out_features=2, bias=True)
Parameter containing:
tensor([[ 0.2302,  0.7597, -0.7996],
        [ 1.2362, -1.2230,  0.7273]], requires_grad=True)
Parameter containing:
tensor([ 0.9590, -0.3395], requires_grad=True)


In [27]:
# Pass in our batch to this layaer
y_linear = linear(X)
print(y_linear)

tensor([[ 1.0998,  0.6727],
        [ 1.3834,  0.9116],
        [ 0.3924, -0.9301],
        [ 0.2468,  0.1703],
        [ 1.2715, -2.7682]], grad_fn=<AddmmBackward0>)


In [28]:
y_manual = y # (from earlier)
torch.allclose(y_manual, y_linear)

True

### Step 4: Shape semantics check

Answer these in this Markdown cell after running the code:

1. What does each row of `X` represent? Each column?
2. What does each row of `W` represent in your convention? Each column?
3. In PyTorch's `nn.Linear` convention, what does each row of `linear.weight` represent?
4. For one output row `Y[i]`, which rows of `X` and which columns of `W` contribute? With what weights?

Write your answers here.

Each row of X is our 3d vector. A vector being a direction in some space (3d) represented by the scalars on each basis vector of that space (e1, e2, e3).
Each column of X is the scalar of that particular basis vector dimension for the row's vector.

W each row represents the image vector's basis in that dimension. Each column represents the full input length image vector to map an input vector to the output in this sapce. 
In this case the column is the amount of each basis vector from input space to get to that dimension in output space. Column 1 is 3 values to map the 3 input values to e1 in output space.

In nn.Linear W is transposed. Where now each row is a neuron. Meaning the 3 values in row 1 map a 3d input vector to e1 in the new space. Where the input is weighted sum + a bias term and output is the neuron value. We map the input vector to a single dimension in our output space.

For Y[i] we take the X[i] row as our input vector. Then given the W matrix as 3 rows by 2 columns, you take X[i,j] as the weight across W[i] (input vector is saying much of this input dimension contributes to the new map). Then you sum down all the columns to get Y[i].

## Finish

Paste back your by-hand matrices and your `nn.Linear without nn.Linear` results. Include your shape comments and the result of `torch.allclose`.